# Dataset summary

**Region reference:** [BC Government - Fire Centres](https://www2.gov.bc.ca/gov/content/safety/wildfire-status/about-bcws/fire-centres) (official descriptions of the six regions).

| Item | Details |
| --- | --- |
| Source | NASA FIRMS, VIIRS S-NPP |
| Area and season | British Columbia; May-September 2021-2023 |
| Size | **522,167 rows; 15 columns** |
| Null values | **0** in the original columns |
| Rows by year | 2021: 128,440; 2022: 17,077; 2023: 376,650 |
| Test data | 2024 kept separate |
| Filters applied | BC boundary and season only |

**One row = one hotspot observation, not a unique fire or confirmed ignition.** Repeated detections are expected; no nulls does not prove complete satellite coverage.

### Columns

Examples are from the first data row. [NASA field reference](https://firms.modaps.eosdis.nasa.gov/content/descriptions/FIRMS_VIIRS_Firehotspots.html).

| Column | Meaning | Example |
| --- | --- | --- |
| `latitude` | Pixel-centre latitude; degrees north | `50.29224` |
| `longitude` | Pixel-centre longitude; negative = west | `-119.18909` |
| `brightness` | I4 brightness temperature (K), not air temperature | `299.39` |
| `scan` | Across-scan pixel width (km) | `0.39` |
| `track` | Along-track pixel length (km) | `0.44` |
| `acq_date` | Observation date, not ignition date | `2021-05-01` |
| `acq_time` | UTC time, HHMM; 1035 = 10:35 | `1035` |
| `satellite` | Satellite platform: Suomi NPP | `SNPP` |
| `instrument` | Sensor field; export says SNPP despite VIIRS source | `SNPP` |
| `confidence` | l = low; n = nominal; h = high | `n` |
| `version` | Collection 2, standard processing | `2` |
| `bright_t31` | Second brightness temperature (K); confirm I5 alias | `278.12` |
| `frp` | Pixel fire radiative power (MW) | `0.63` |
| `daynight` | D = day; N = night | `N` |
| `type` | Inferred source: 0 = vegetation; 1 = volcano; 2 = other fixed land source; 3 = water | `0` |

# Study-region comparison

**Goal:** compare hotspot activity, station distance, and weather completeness across candidate regions.

**Scope:** development years only; region, grid size, and prediction target remain open.

In [3]:
# Dataset overview: BC hotspots, May-September 2021-2023 (development data only)
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
hotspots = pd.read_csv(ROOT / 'data/exploration/bc_may_sep_2021_2023.csv')
print(f'Total rows: {len(hotspots):,}')
print(f'Total columns: {len(hotspots.columns)}')
print('Columns:')
display(pd.DataFrame({'column': hotspots.columns}))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/williamting/Desktop/rbc-team-gaw/notebooks/data/exploration/bc_may_sep_2021_2023.csv'

In [ ]:
# Null-value check on the original columns, before adding derived features
null_counts = hotspots.isna().sum()
print(f'Any null values: {bool(null_counts.any())}')
print(f'Total null values: {int(null_counts.sum()):,}')
display(pd.DataFrame({
    'null_count': null_counts,
    'null_percent': (100 * hotspots.isna().mean()).round(2),
}).rename_axis('column'))

In [4]:
from pathlib import Path
import os, sys, json, hashlib
from datetime import datetime, timezone
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/exploration/bc_may_sep_2021_2023.csv').exists(), 'Open from project root or notebooks directory; run initial FIRMS exploration first.'
CACHE = ROOT / 'data/region_exploration'
CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE / 'matplotlib-cache'))
sys.path.insert(0, str(ROOT / 'data/check-dependencies'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from shapely.geometry import shape
from shapely import intersects_xy, prepare
from sklearn.neighbors import BallTree
from IPython.display import display, Markdown
pd.set_option('display.max_columns', 20)
YEARS = [2021, 2022, 2023]
MONTHS = [5, 6, 7, 8, 9]
EXPECTED = pd.DatetimeIndex([d for y in YEARS for d in pd.date_range(f'{y}-05-01', f'{y}-09-30')])
assert len(EXPECTED) == 459
session = requests.Session()
session.mount('https://', HTTPAdapter(max_retries=Retry(total=2, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])))
def cached_json(name, url, params):
    path = CACHE / name
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    response = session.get(url, params=params, timeout=60)
    response.raise_for_status()
    obj = response.json()
    if 'error' in obj or 'features' not in obj:
        raise ValueError(f'Unexpected response from {response.url}')
    if obj.get('exceededTransferLimit') or any(link.get('rel') == 'next' for link in obj.get('links', [])):
        raise ValueError('Response is paginated/truncated: increase the limit or implement pagination before using it.')
    if obj.get('numberMatched', len(obj['features'])) != len(obj['features']):
        raise ValueError('Incomplete response')
    path.write_text(json.dumps(obj), encoding='utf-8')
    (CACHE / (name + '.source.json')).write_text(json.dumps({
        'url': response.url, 'retrieved_utc': datetime.now(timezone.utc).isoformat(),
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}, indent=2), encoding='utf-8')
    return obj
print('Expected days per station:', len(EXPECTED), '(153 per summer)')

AssertionError: Open from project root or notebooks directory; run initial FIRMS exploration first.

## 1. Hotspots by fire centre

- Compare all six regions using current [DataBC boundaries](https://delivery.maps.gov.bc.ca/arcgis/rest/services/whse/bcgw_pub_whse_legal_admin_boundaries/MapServer/1), simplified to 0.002 degrees.
- Invalid polygons are repaired and reported; shared-edge matches go to the alphabetically first region.
- Counts include repeat detections and are not adjusted for area or observation coverage. Validate exact boundaries before modeling.

In [ ]:
CENTRE_URL = 'https://delivery.maps.gov.bc.ca/arcgis/rest/services/whse/bcgw_pub_whse_legal_admin_boundaries/MapServer/1/query'
regions_json = cached_json('fire_centres.geojson', CENTRE_URL, {
    'where': '1=1', 'outFields': '*', 'outSR': 4326, 'f': 'geojson', 'maxAllowableOffset': 0.002})
assert len(regions_json['features']) == 6 and not regions_json.get('exceededTransferLimit')
REGIONS = {f['properties']['MOF_FIRE_CENTRE_NAME'].replace(' Fire Centre', ''): shape(f['geometry'])
           for f in regions_json['features']}
REGIONS = dict(sorted(REGIONS.items()))
from shapely import make_valid
from shapely.validation import explain_validity
from shapely.ops import unary_union
geometry_repairs = []
def polygon_parts(g):
    if g.geom_type == 'Polygon':
        return [g]
    return [part for child in getattr(g, 'geoms', []) for part in polygon_parts(child)]
for name, geometry in REGIONS.items():
    if not geometry.is_valid:
        geometry_repairs.append({'region': name, 'reason': explain_validity(geometry)})
        geometry = unary_union(polygon_parts(make_valid(geometry)))
        REGIONS[name] = geometry
    assert geometry.is_valid and not geometry.is_empty
    prepare(geometry)
print('Geometry repairs (make_valid; polygon components retained):', geometry_repairs)
def assign_region(frame):
    hits = np.column_stack([intersects_xy(g, frame.longitude.to_numpy(), frame.latitude.to_numpy()) for g in REGIONS.values()])
    names = np.array(list(REGIONS))
    labels = np.where(hits.any(axis=1), names[hits.argmax(axis=1)], 'Unassigned')
    return labels, hits.sum(axis=1)
hotspots = pd.read_csv(ROOT / 'data/exploration/bc_may_sep_2021_2023.csv')
hotspots['date'] = pd.to_datetime(hotspots.acq_date)
assert hotspots.date.dt.year.isin(YEARS).all()
assert hotspots.date.dt.month.isin(MONTHS).all()
hotspots['year'] = hotspots.date.dt.year
hotspots['region'], matches = assign_region(hotspots)
display(pd.Series({'observations': len(hotspots), 'unassigned_polygon_edges': int((matches == 0).sum()),
                   'multiple_region_matches': int((matches > 1).sum())}).to_frame('count'))
regional_counts = pd.crosstab(hotspots.region, hotspots.year).reindex(columns=YEARS, fill_value=0)
regional_counts['total'] = regional_counts[YEARS].sum(axis=1)
regional_counts['smallest_year'] = regional_counts[YEARS].min(axis=1)
regional_counts['2023_share_pct'] = (100 * regional_counts[2023] / regional_counts.total).round(1)
assert regional_counts.total.sum() == len(hotspots)
display(regional_counts.sort_values('smallest_year', ascending=False))
regional_counts.to_csv(CACHE / 'regional_hotspot_counts.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
regional_counts.reindex(list(REGIONS))[YEARS].plot.bar(ax=ax, color=['#377eb8', '#ff9e3d', '#3b9b61'])
ax.set(title='Summer hotspot observations by region and year', ylabel='Observations', xlabel='')
ax.tick_params(axis='x', rotation=20)
def format_count(value):
    return f'{value / 1000:.1f}'.rstrip('0').rstrip('.') + 'K' if value >= 1000 else str(int(value))
for bars in ax.containers:
    ax.bar_label(bars, labels=[format_count(bar.get_height()) for bar in bars],
                 padding=3, fontsize=8)
ax.margins(y=0.15)
from matplotlib.ticker import FuncFormatter
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: format_count(value)))
plt.tight_layout()
plt.show()

## 2. Candidate regions and stations

- **Shortlist:** three regions with the highest counts in their least-active development year.
- **Station candidates:** [ECCC inventory](https://api.weather.gc.ca/collections/climate-stations?f=html) date ranges span the study period.
- Operating dates do not guarantee complete measurements.

In [ ]:
STATIONS_URL = 'https://api.weather.gc.ca/collections/climate-stations/items'
inventory = cached_json('stations.json', STATIONS_URL, {'f': 'json', 'limit': 10000, 'PROV_STATE_TERR_CODE': 'BC'})
assert inventory['numberMatched'] == len(inventory['features'])
stations = pd.DataFrame([dict(f['properties'], longitude=f['geometry']['coordinates'][0],
                              latitude=f['geometry']['coordinates'][1]) for f in inventory['features']])
assert stations.PROV_STATE_TERR_CODE.eq('BC').all()
for col in ['DLY_FIRST_DATE', 'DLY_LAST_DATE', 'HLY_FIRST_DATE', 'HLY_LAST_DATE']:
    stations[col] = pd.to_datetime(stations[col], errors='coerce')
stations['region'], station_matches = assign_region(stations)
stations['daily_spans_study'] = (stations.DLY_FIRST_DATE <= EXPECTED.min()) & (stations.DLY_LAST_DATE >= EXPECTED.max())
stations['hourly_spans_study'] = (stations.HLY_FIRST_DATE <= EXPECTED.min()) & (stations.HLY_LAST_DATE >= EXPECTED.max())
eligible = stations.loc[stations.daily_spans_study & stations.region.isin(REGIONS)].copy()
assert not eligible.CLIMATE_IDENTIFIER.duplicated().any()
inventory_summary = eligible.groupby('region').agg(
    daily_span_candidates=('CLIMATE_IDENTIFIER', 'size'), hourly_span_candidates=('hourly_spans_study', 'sum'))
display(regional_counts.join(inventory_summary).sort_values('smallest_year', ascending=False))
CANDIDATES = regional_counts.loc[list(REGIONS)].nlargest(3, 'smallest_year').index.tolist()
STATIONS_PER_REGION = 3
print('Initial comparison regions:', ', '.join(CANDIDATES))
print('Unassigned station coordinates:', int((station_matches == 0).sum()))

### Station distance and sampling

- Measure great-circle distances from hotspots to all candidate stations within each region.
- Sample **three stations per region**: nearest the mean hotspot location, then spread selections geographically.
- Repeated hotspots receive more weight. This sample does not establish full-area or elevation coverage.

In [ ]:
def nearest_km(points, reference):
    tree = BallTree(np.radians(reference[['latitude', 'longitude']].to_numpy()), metric='haversine')
    distances, _ = tree.query(np.radians(points[['latitude', 'longitude']].to_numpy()), k=1)
    return distances[:, 0] * 6371.0088
chosen = []
proximity_rows = []
for region in CANDIDATES:
    points = hotspots.loc[hotspots.region == region]
    pool = eligible.loc[eligible.region == region].sort_values('CLIMATE_IDENTIFIER').copy()
    assert len(pool) > 0, f'No inventory candidates for {region}'
    distances = nearest_km(points, pool)
    proximity_rows.append({'region': region, 'inventory_candidates': len(pool),
        'median_distance_km': np.median(distances), 'p90_distance_km': np.quantile(distances, .9),
        'within_25km_pct': 100 * (distances <= 25).mean()})
    centre = points[['latitude', 'longitude']].mean().to_frame().T
    first = pool.iloc[int(np.argmin(nearest_km(pool, centre)))].name
    selected_indices = [first]
    while len(selected_indices) < min(STATIONS_PER_REGION, len(pool)):
        remaining = pool.drop(index=selected_indices)
        idx = remaining.iloc[int(np.argmax(nearest_km(remaining, pool.loc[selected_indices])))].name
        selected_indices.append(idx)
    chosen.append(pool.loc[selected_indices])
sample_stations = pd.concat(chosen, ignore_index=True)
proximity = pd.DataFrame(proximity_rows).set_index('region')
display(proximity.round(1))
display(sample_stations[['region', 'STATION_NAME', 'CLIMATE_IDENTIFIER', 'latitude', 'longitude',
                         'ELEVATION', 'hourly_spans_study']])
sample_stations.to_csv(CACHE / 'sampled_stations.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, len(CANDIDATES), figsize=(16, 6), squeeze=False)
for ax, region in zip(axes[0], CANDIDATES):
    geometry = REGIONS[region]
    for polygon in (geometry.geoms if geometry.geom_type == 'MultiPolygon' else [geometry]):
        ax.plot(*polygon.exterior.xy, color='gray', linewidth=.7)
    points = hotspots.loc[hotspots.region == region]
    ax.hexbin(points.longitude, points.latitude, gridsize=40, mincnt=1, bins='log', cmap='Oranges')
    pool = eligible.loc[eligible.region == region]
    picked = sample_stations.loc[sample_stations.region == region]
    ax.scatter(pool.longitude, pool.latitude, s=15, color='#3268ad', label='Inventory candidate')
    ax.scatter(picked.longitude, picked.latitude, s=75, marker='*', color='black', label='Audited sample')
    ax.set(title=region, xlabel='Longitude', ylabel='Latitude')
    ax.set_aspect(1 / np.cos(np.deg2rad(points.latitude.mean())))
    ax.legend(fontsize=8)
fig.suptitle('Hotspot locations and candidate weather stations (density colours scaled per panel)')
plt.tight_layout()
plt.show()

## 3. Actual daily weather

Source: [ECCC daily observations](https://api.weather.gc.ca/collections/climate-daily?f=html) and [field definitions](https://api.weather.gc.ca/collections/climate-daily/schema?f=html).

- **459 expected summer days per station**; absent dates and null values count as unavailable.
- API coverage is a station subset; other archives may contain additional data.
- Humidity = daily minimum; wind = maximum gust, not sustained wind.

In [ ]:
DAILY_URL = 'https://api.weather.gc.ca/collections/climate-daily/items'
weather_parts = []
download_audit = []
for row in sample_stations.itertuples():
    climate_id = str(row.CLIMATE_IDENTIFIER)
    obj = cached_json(f'daily_{climate_id}_2021_2023.json', DAILY_URL, {
        'f': 'json', 'limit': 2000, 'CLIMATE_IDENTIFIER': climate_id,
        'datetime': '2021-05-01T00:00:00Z/2023-09-30T23:59:59Z'})
    assert obj.get('numberMatched', len(obj['features'])) == len(obj['features'])
    records = pd.DataFrame([f['properties'] for f in obj['features']])
    download_audit.append({'station': row.STATION_NAME, 'returned_rows': len(records)})
    if not records.empty:
        assert records.CLIMATE_IDENTIFIER.astype(str).eq(climate_id).all(), 'API ignored station filter'
        records['date'] = pd.to_datetime(records.LOCAL_DATE).dt.normalize()
        assert records.date.between('2021-05-01', '2023-09-30').all(), 'API ignored date filter'
        assert not records.date.duplicated().any(), 'Resolve duplicate station/date records before profiling'
        records = records.loc[records.date.isin(EXPECTED)].copy()
    else:
        records = pd.DataFrame(columns=['date'])
    records = records.set_index('date').reindex(EXPECTED).rename_axis('date').reset_index()
    records['climate_id'] = climate_id
    records['station'] = row.STATION_NAME
    records['region'] = row.region
    weather_parts.append(records)
weather = pd.concat(weather_parts, ignore_index=True)
assert len(weather) == len(sample_stations) * len(EXPECTED)
assert weather.date.dt.year.isin(YEARS).all()
display(pd.DataFrame(download_audit))
print('Expected station-days audited:', len(weather))

## 4. Completeness by station and year

| Measure | Definition |
| --- | --- |
| Core weather | Maximum temperature and precipitation both available |
| Provisional pass | Core weather on at least **90% of days in every summer** |
| Longest gap | Consecutive missing core days within a summer |

Availability is not a quality check. Flags, units, trace precipitation, station changes, and release timing still need review.

In [ ]:
FIELDS = ['MAX_TEMPERATURE', 'TOTAL_PRECIPITATION', 'MIN_REL_HUMIDITY', 'SPEED_MAX_GUST']
for field in FIELDS:
    weather[field] = pd.to_numeric(weather.get(field, pd.Series(np.nan, index=weather.index)), errors='coerce')
weather['core_present'] = weather[['MAX_TEMPERATURE', 'TOTAL_PRECIPITATION']].notna().all(axis=1)
weather['year'] = weather.date.dt.year
def longest_missing(values):
    longest = run = 0
    for available in values:
        run = 0 if available else run + 1
        longest = max(longest, run)
    return longest
coverage_rows = []
for (region, station, climate_id, year), frame in weather.groupby(['region', 'station', 'climate_id', 'year']):
    frame = frame.sort_values('date')
    assert len(frame) == 153
    item = {'region': region, 'station': station, 'climate_id': climate_id, 'year': year,
            'core_pct': 100 * frame.core_present.mean(),
            'longest_core_gap_days': longest_missing(frame.core_present)}
    item.update({field + '_pct': 100 * frame[field].notna().mean() for field in FIELDS})
    coverage_rows.append(item)
coverage = pd.DataFrame(coverage_rows)
display(coverage.round(1))
coverage.to_csv(CACHE / 'sample_station_year_coverage.csv', index=False)
flag_rows = []
for field in FIELDS:
    flag = field + '_FLAG'
    if flag in weather:
        counts = weather[flag].fillna('(blank)').astype(str).value_counts()
        flag_rows.extend({'field': field, 'flag': key, 'station_days': count} for key, count in counts.items())
display(pd.DataFrame(flag_rows))
MIN_CORE_PCT = 90
station_audit = coverage.groupby(['region', 'station', 'climate_id']).agg(
    worst_summer_core_pct=('core_pct', 'min'), longest_core_gap_days=('longest_core_gap_days', 'max')).reset_index()
station_audit['passes_screen'] = station_audit.worst_summer_core_pct >= MIN_CORE_PCT
display(station_audit.round(1))

In [ ]:
monthly = weather.assign(month=weather.date.dt.strftime('%Y-%m')).groupby(['station', 'month']).core_present.mean().unstack() * 100
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(monthly, aspect='auto', vmin=0, vmax=100, cmap='RdYlGn')
ax.set_yticks(range(len(monthly)), monthly.index)
ax.set_xticks(range(len(monthly.columns)), monthly.columns, rotation=60, ha='right')
ax.set_title('Days with both maximum temperature and precipitation (%)')
fig.colorbar(im, ax=ax, label='% of expected days')
plt.tight_layout()
plt.show()

## 5. Candidate comparison

- Distinguish distances to **all inventory candidates** from distances to **sampled stations passing the screen**.
- A day with any core weather means at least one sampled station has both fields, not coverage of every grid cell.
- Nearby stations may still poorly represent mountain conditions.

In [ ]:
sample_summary = station_audit.groupby('region').agg(
    stations_sampled=('station', 'size'), sampled_stations_passing=('passes_screen', 'sum'),
    worst_sample_summer_core_pct=('worst_summer_core_pct', 'min'))
any_core = weather.groupby(['region', 'date']).core_present.any().groupby('region').mean() * 100
comparison = regional_counts.loc[CANDIDATES, YEARS + ['smallest_year', '2023_share_pct']].join(proximity).join(sample_summary)
comparison['days_with_any_sampled_core_pct'] = any_core
for region in CANDIDATES:
    good_ids = station_audit.loc[(station_audit.region == region) & station_audit.passes_screen, 'climate_id']
    good = sample_stations.loc[sample_stations.CLIMATE_IDENTIFIER.astype(str).isin(good_ids)]
    points = hotspots.loc[hotspots.region == region]
    comparison.loc[region, 'p90_km_to_screened_sample'] = np.quantile(nearest_km(points, good), .9) if len(good) else np.nan
display(comparison.round(1))
comparison.to_csv(CACHE / 'candidate_region_comparison.csv')
manifest = {
    'analysis_years': YEARS, 'months': MONTHS, 'candidates': CANDIDATES,
    'stations_per_region': STATIONS_PER_REGION, 'core_screen_pct': MIN_CORE_PCT,
    'holdout_loaded': False,
    'source_urls': {'regions': CENTRE_URL, 'stations': STATIONS_URL, 'daily': DAILY_URL},
    'cache_hashes': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                     for p in CACHE.glob('*.json') if p.name != 'analysis_manifest.json'},
    'boundary_sha256': hashlib.sha256((CACHE / 'fire_centres.geojson').read_bytes()).hexdigest(),
    'firms_development_sha256': hashlib.sha256((ROOT / 'data/exploration/bc_may_sep_2021_2023.csv').read_bytes()).hexdigest(),
    'versions': {name: __import__(name).__version__ for name in ['pandas', 'numpy', 'matplotlib', 'shapely', 'sklearn']}
}
(CACHE / 'analysis_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Checks passed: development-only dates, complete API responses, unique station-dates, expected-date denominators, and reconciled hotspot counts.')

## 6. Region detail

Monthly hotspot counts and sampled-station completeness for the selected region.

In [ ]:
import ipywidgets as widgets
def explore_region(region):
    region_points = hotspots.loc[hotspots.region == region].copy()
    region_points['month'] = region_points.date.dt.month
    counts = region_points.groupby(['month', 'year']).size().unstack(fill_value=0).reindex(index=MONTHS, columns=YEARS, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 3))
    counts.plot(ax=ax, marker='o')
    ax.set(title=f'{region}: monthly hotspot observations', xlabel='Month', ylabel='Observations',
           xticks=MONTHS, xticklabels=['May', 'Jun', 'Jul', 'Aug', 'Sep'])
    plt.tight_layout()
    plt.show()
    display(station_audit.loc[station_audit.region == region].round(1))
explore_region(CANDIDATES[0])
selector = widgets.Dropdown(options=CANDIDATES, description='Region:')
display(widgets.VBox([selector, widgets.interactive_output(explore_region, {'region': selector})]))

## Limitations

- FIRMS provides satellite detection dates, not verified ignition dates.
- Repeated detections are not independent fires; long-lasting fires can dominate counts.
- No detection does not prove no fire because satellite coverage is incomplete.
- 2023 contributes 72.13% of development observations; more years are needed for seasonal variability.
- Region, grid size, target, prediction horizon, and negative-label rules remain open.
- Weather stations are unevenly distributed, and the current audit covers only a small sample.
- Weather units, timestamps, quality flags, and station coverage need validation before joining.
- 2024 remains reserved for final testing.